In [5]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [16]:
import json
import folium

# 1) Carrega seu JSON com os amigos
with open("amigos.json", "r", encoding="utf-8") as f:
    amigos = json.load(f)

# 2) Filtra só quem é do Brasil e tem coordenadas
amigos_br = [
    a for a in amigos
    if a.get("pais") == "Brasil" and "lat" in a and "lon" in a
]

if not amigos_br:
    raise ValueError("Nenhum amigo do Brasil com lat/lon encontrado no JSON!")

# 3) Calcula o centro aproximado do grupo (média das coords)
lat_media = sum(a["lat"] for a in amigos_br) / len(amigos_br)
lon_media = sum(a["lon"] for a in amigos_br) / len(amigos_br)

# 4) Cria o mapa centralizado na galera do BR
m = folium.Map(location=[lat_media, lon_media], zoom_start=12)

# 5) Adiciona marcadores para cada amigo
for a in amigos_br:
    folium.Marker(
        location=[a["lat"], a["lon"]],
        popup=f"{a['nome']} — {a['endereco']}",
        tooltip=a["nome"],
    ).add_to(m)

# (Opcional) Marca o ponto médio também
folium.CircleMarker(
    location=[lat_media, lon_media],
    radius=8,
    color="red",
    fill=True,
    fill_opacity=0.7,
    popup="Centro aproximado do grupo (BR)"
).add_to(m)

# 6) Salva o HTML
m.save("mapa_amigos_br.html")
print("Mapa criado: mapa_amigos_br.html")

Mapa criado: mapa_amigos_br.html


In [17]:
# Mostrar a latitude e longitude do centro aproximado
print("Centro aproximado do grupo (BR):")
print("Latitude :", lat_media)
print("Longitude:", lon_media)


Centro aproximado do grupo (BR):
Latitude : -23.587515633333332
Longitude: -46.66782004444445


In [18]:
import json
import folium

# 1) Carrega seu JSON com os amigos
with open("amigos.json", "r", encoding="utf-8") as f:
    amigos = json.load(f)

# 2) Filtra todos com coordenadas (Brasil + NYC)
amigos_todos = [
    a for a in amigos
    if "lat" in a and "lon" in a
]

# 3) Filtra só Brasil (para o mapa SP)
amigos_br = [
    a for a in amigos_todos
    if a.get("pais") == "Brasil"
]

if not amigos_br:
    raise ValueError("Nenhum amigo do Brasil com lat/lon encontrado no JSON!")

# 4) Centro aproximado do grupo BR
lat_media_br = sum(a["lat"] for a in amigos_br) / len(amigos_br)
lon_media_br = sum(a["lon"] for a in amigos_br) / len(amigos_br)

# 5) Centro aproximado do grupo GLOBAL (incluindo NYC)
lat_media_global = sum(a["lat"] for a in amigos_todos) / len(amigos_todos)
lon_media_global = sum(a["lon"] for a in amigos_todos) / len(amigos_todos)

print("Centro BR:")
print(f"  lat = {lat_media_br}")
print(f"  lon = {lon_media_br}")

print("\nCentro Global (inclui NYC):")
print(f"  lat = {lat_media_global}")
print(f"  lon = {lon_media_global}")

# 6) Cria um mapa centralizado na galera do BR
m = folium.Map(location=[lat_media_br, lon_media_br], zoom_start=12)

# 7) Marcadores do BR
for a in amigos_br:
    folium.Marker(
        location=[a["lat"], a["lon"]],
        popup=f"{a['nome']} — {a['endereco']}",
        tooltip=a["nome"],
    ).add_to(m)

# 8) Ponto médio BR
folium.CircleMarker(
    location=[lat_media_br, lon_media_br],
    radius=8,
    color="red",
    fill=True,
    fill_opacity=0.7,
    popup="Centro BR"
).add_to(m)

# 9) Adiciona o Leo Gomes (NYC) no mapa (com cor especial)
leo = next(a for a in amigos if a["nome"] == "Leo Gomes")

folium.Marker(
    location=[leo["lat"], leo["lon"]],
    popup="Leo Gomes — NYC",
    tooltip="Leo Gomes",
    icon=folium.Icon(color="green")
).add_to(m)

# 10) Adiciona o centro global no mapa
folium.CircleMarker(
    location=[lat_media_global, lon_media_global],
    radius=10,
    color="purple",
    fill=True,
    fill_opacity=0.5,
    popup="Centro Global"
).add_to(m)

# 11) Salva o HTML
m.save("mapa_amigos_BR_NYC.html")
print("Mapa criado: mapa_amigos_BR_NYC.html")


Centro BR:
  lat = -23.587515633333332
  lon = -46.66782004444445

Centro Global (inclui NYC):
  lat = -20.20057150526316
  lon = -48.1059554
Mapa criado: mapa_amigos_BR_NYC.html


In [21]:
import json
import folium
from math import radians, sin, cos, asin, sqrt, atan2, degrees

# --- Função de distância geodésica (Haversine) em km ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # raio médio da Terra em km

    lat1_rad, lon1_rad = radians(lat1), radians(lon1)
    lat2_rad, lon2_rad = radians(lat2), radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = sin(dlat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon / 2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# --- Função de ponto médio geográfico entre 2 coordenadas ---
def midpoint(lat1, lon1, lat2, lon2):
    # converte pra radianos
    lat1_rad, lon1_rad = radians(lat1), radians(lon1)
    lat2_rad, lon2_rad = radians(lat2), radians(lon2)

    # converte pra coordenadas cartesianas 3D
    x1, y1, z1 = cos(lat1_rad) * cos(lon1_rad), cos(lat1_rad) * sin(lon1_rad), sin(lat1_rad)
    x2, y2, z2 = cos(lat2_rad) * cos(lon2_rad), cos(lat2_rad) * sin(lon2_rad), sin(lat2_rad)

    # média dos vetores
    xm, ym, zm = (x1 + x2) / 2, (y1 + y2) / 2, (z1 + z2) / 2

    # volta pra lat/lon
    hyp = sqrt(xm * xm + ym * ym)
    latm = atan2(zm, hyp)
    lonm = atan2(ym, xm)

    return degrees(latm), degrees(lonm)

# 1) Carrega seu JSON com os amigos
with open("amigos.json", "r", encoding="utf-8") as f:
    amigos = json.load(f)

# 2) Filtra só quem é do Brasil e tem coordenadas
amigos_br = [
    a for a in amigos
    if a.get("pais") == "Brasil" and "lat" in a and "lon" in a
]

if not amigos_br:
    raise ValueError("Nenhum amigo do Brasil com lat/lon encontrado no JSON!")

# 3) Calcula o centro aproximado do grupo (média das coords) - SÓ BR
lat_media = sum(a["lat"] for a in amigos_br) / len(amigos_br)
lon_media = sum(a["lon"] for a in amigos_br) / len(amigos_br)

print("Centro aproximado do grupo (BR):")
print(f"  lat = {lat_media}")
print(f"  lon = {lon_media}")

# 4) Pega o Leo em NYC
leo = next(a for a in amigos if a["nome"] == "Leo Gomes")

dist_nyc_centro_br = haversine(lat_media, lon_media, leo["lat"], leo["lon"])
print(f"\nDistância NYC ↔ centro BR ≈ {dist_nyc_centro_br:.1f} km")

# 5) Calcula o 'destino justo' = ponto médio entre centro BR e NYC
lat_meio, lon_meio = midpoint(lat_media, lon_media, leo["lat"], leo["lon"])
print("\nPonto médio geográfico (destino justo teórico):")
print(f"  lat = {lat_meio}")
print(f"  lon = {lon_meio}")

# 6) Cria o mapa centralizado na galera do BR
m = folium.Map(location=[lat_media, lon_media], zoom_start=3)  # zoom mais aberto pra ver o meio do caminho

# 7) Adiciona marcadores para cada amigo do BR
for a in amigos_br:
    folium.Marker(
        location=[a["lat"], a["lon"]],
        popup=f"{a['nome']} — {a['endereco']}",
        tooltip=a["nome"],
    ).add_to(m)

# 8) Marca o ponto médio BR
folium.CircleMarker(
    location=[lat_media, lon_media],
    radius=8,
    color="red",
    fill=True,
    fill_opacity=0.7,
    popup="Centro aproximado do grupo (BR)"
).add_to(m)

# 9) Adiciona o Leo (NYC) no mapa (em verde)
folium.Marker(
    location=[leo["lat"], leo["lon"]],
    popup=f"Leo Gomes — NYC (dist ≈ {dist_nyc_centro_br:.0f} km do centro BR)",
    tooltip="Leo Gomes",
    icon=folium.Icon(color="green")
).add_to(m)

# 10) Adiciona o ponto médio geográfico (destino justo teórico)
folium.Marker(
    location=[lat_meio, lon_meio],
    popup="Ponto médio BR ↔ NYC (destino justo teórico)",
    tooltip="Meio do caminho",
    icon=folium.Icon(color="purple", icon="info-sign")
).add_to(m)

# 11) Salva o HTML
m.save("mapa_destino_justo.html")
print("Mapa criado: mapa_destino_justo.html")


Centro aproximado do grupo (BR):
  lat = -23.587515633333332
  lon = -46.66782004444445

Distância NYC ↔ centro BR ≈ 7692.7 km

Ponto médio geográfico (destino justo teórico):
  lat = 8.832332634657504
  lon = -59.00697313615481
Mapa criado: mapa_destino_justo.html


In [20]:
from geopy.geocoders import Nominatim
import folium

# 🏠 1) Defina aqui o endereço que você quer marcar
endereco = "Rua Teviot 101, São Paulo, Brasil"

# 2) Geocoding com Nominatim (OpenStreetMap)
geolocator = Nominatim(user_agent="mapa-amigos/1.0 (seu-email@exemplo.com)")
location = geolocator.geocode(endereco, timeout=10)

if location is None:
    raise ValueError(f"Endereço não encontrado: {endereco}")

lat, lon = location.latitude, location.longitude
print(f"Endereço geocodificado:\n  {endereco}\n  lat={lat}, lon={lon}")

# 3) Cria o mapa centralizado no endereço
m = folium.Map(location=[lat, lon], zoom_start=16)

# 4) Adiciona o pin (marcador)
folium.Marker(
    location=[lat, lon],
    popup=endereco,
    tooltip="Destino escolhido"
).add_to(m)

# (Opcional) destacar com um círculo em volta
folium.CircleMarker(
    location=[lat, lon],
    radius=12,
    color="red",
    fill=True,
    fill_opacity=0.4
).add_to(m)

# 5) Mostrar o mapa no notebook
m

# (Opcional) salvar em HTML
#m.save("pin_endereco.html")


Endereço geocodificado:
  Rua Teviot 101, São Paulo, Brasil
  lat=-23.5866276, lon=-46.6689286
